## Read calibration data
1. The standard folders has to named of the molecule and pubchem id e.g. Ado 60961
2. Write all molecules with their retention time in retention_times
3. Change the calib_path to where the standards are saved and date
4. run the script



In [ ]:
from chromhandler import ChromAnalyzer, Molecule, Protein
from pathlib import Path
import os

# define the retention times in a dictionary, then we can access them by the molecule name from the file names
retention_times = {
    "Ado": 0.78,
    "ADP": 3.79,
    "AMP": 1.96,
    "ATP": 6.12,
    "Gua": 0.85,
    "GMP": 2.04,
    "GDP": 4.98,
    "GTP": 8.39,
}
# define the date of the calibration data
date = "20251016"

# Define path to the calibration data
calib_path = Path("data/"+date+"_standards")
wavelength = 254  # nm
print("Loading calibration data from:", calib_path)


Loading calibration data from: data\20251016_standards


In [4]:

# loop over all the folders in the calibration data
for molecule_path in calib_path.iterdir():
    name_and_id_list = molecule_path.name.split(" ")
    name = name_and_id_list[0]
    pubchem_cid = int(name_and_id_list[1])
    path = str(molecule_path)

    # read the calibration data
    analyzer = ChromAnalyzer.read_agilent(
        path=path,
        ph=8,
        temperature=37,
        mode="calibration",
    )

    # define the molecules
    molecule = analyzer.define_molecule(
        pubchem_cid=pubchem_cid,
        id=name,
        retention_time=retention_times[name],
        retention_tolerance=0.4,
        wavelength=wavelength,
    )
    print(f"Processing molecule: {molecule.id} with PubChem CID: {molecule.pubchem_cid}")
    # run calibrator
    analyzer.add_standard(molecule=molecule, wavelength=wavelength, visualize=True)
    savepath = "result/calib_molecules"+date+"standards"
    # create directory if it does not exist
    if not os.path.exists(savepath):
        os.makedirs(savepath)
    # save molecule including the fitted calibration model to file
    molecule.save_json(os.path.join(savepath, f"{molecule.id}.json"))

Processing molecule: Ado with PubChem CID: 60961


AssertionError: Number of Adenosine peak areas 0 and concentrations 8 do not match.

## Add molecules without calibration

In [ ]:
TrisHCL = Molecule(
    id="TrisHCL",
    name="Tris HCL",
    pubchem_cid=93573,
)

MgCl2 = Molecule(
    id="MgCl2",
    name="Magnesium Chloride",
    pubchem_cid=5360315,
)

# save to file
for molecule in [TrisHCL, MgCl2]:
    path = f"result/calib_molecules/{molecule.id}.json"
    molecule.save_json(path)
    print(f"Saved {molecule.id} to {path}")

Saved TrisHCL to result/calib_molecules/TrisHCL.json
Saved MgCl2 to result/calib_molecules/MgCl2.json


## Create Proteins



In [ ]:
proteins = {
    "EaGK": "O24767",
    "EbPPK": "A0A3D5XRJ5",
}

for name, uniprot_id in proteins.items():
    protein = Protein.from_uniprot(uniprot_id, name=name)
    path = f"result/proteins/{protein.name}.json"
    protein.save_json(path)
    print(f"Saved {protein.id} to {path}")


Saved O24767 to result/proteins/EaGK.json
Saved A0A3D5XRJ5 to result/proteins/EbPPK.json


Calibration is done. You can now proceed with the time samples